In [2]:
"""
=============================================================
 GenAI Smart Travel Planner (Google Colab + Gemini API)
 Updated to use the new `google-genai` SDK and the latest
 Gemini 3.5 Flash model.
=============================================================

WHAT THIS DOES
---------------
Takes destination, budget, number of days, and interests from
the user, then uses Google's Gemini API to generate:
  1. A day-wise itinerary
  2. Suggested attractions
  3. Food recommendations
  4. Practical travel tips

HOW TO RUN (Google Colab)
---------------------------
1. Open https://colab.research.google.com and create a new notebook.
2. Copy this entire file into a single Colab cell (or split into
   multiple cells at the "## CELL" markers below) and run it.
3. You will be prompted to enter a free Gemini API key the first
   time you run it (see instructions below).
4. Follow the on-screen prompts to enter your trip details.

HOW TO GET A FREE GEMINI API KEY
----------------------------------
1. Go to https://aistudio.google.com/app/apikey
2. Sign in with your Google account.
3. Click "Create API key" and copy the generated key.
4. Paste it when this script asks for it (input is hidden using
   getpass so it won't show on screen / won't be saved in the
   notebook file).

## CELL 1: Install the latest Google GenAI SDK ----------------------
# Run this once, in its own Colab cell, so the "!" shell command
# works correctly.
#
# !pip install -q -U google-genai

## CELL 2: Imports and configuration ---------------------------------
from google import genai
from getpass import getpass

MODEL_NAME = "gemini-3.5-flash"


def configure_gemini():
    """
    Prompts the user for their Gemini API key (hidden input) and
    creates a configured genai.Client. Returns the client.
    """
    api_key = getpass("Enter your Gemini API key: ")
    client = genai.Client(api_key=api_key)
    return client


## CELL 3: Collect trip details from the user ------------------------
def get_user_inputs():
    """
    Collects destination, budget, number of days, and interests
    from the user via simple input() prompts.
    """
    print("\n===== Smart Travel Planner: Tell us about your trip =====\n")

    destination = input("Enter your destination (e.g., Goa, Paris, Tokyo): ").strip()

    budget = input("Enter your total budget (e.g., '₹30,000' or '$1500'): ").strip()

    while True:
        days_input = input("Enter number of days for the trip (e.g., 5): ").strip()
        if days_input.isdigit() and int(days_input) > 0:
            days = int(days_input)
            break
        print("Please enter a valid positive number of days.")

    interests = input(
        "Enter your interests, comma-separated "
        "(e.g., history, food, nightlife, nature, shopping, adventure): "
    ).strip()

    return {
        "destination": destination,
        "budget": budget,
        "days": days,
        "interests": interests,
    }


## CELL 4: Build the prompt for Gemini --------------------------------
def build_prompt(trip):
    """
    Builds a detailed, structured prompt so Gemini returns a
    well-organized, day-wise travel plan.
    """
    prompt = f"""
You are an expert travel planner with deep local knowledge.

Create a detailed travel plan for the following trip:
- Destination: {trip['destination']}
- Total Budget: {trip['budget']}
- Duration: {trip['days']} days
- Traveler Interests: {trip['interests']}

Please structure your response in clean Markdown with these sections:

1. **Trip Overview** — A short 2-3 sentence summary of what the
   traveler can expect, tailored to their interests and budget.

2. **Day-Wise Itinerary** — For each day (Day 1 to Day {trip['days']}),
   provide:
   - Morning activity/attraction
   - Afternoon activity/attraction
   - Evening activity/attraction
   - A rough estimated cost for the day (within the overall budget)

3. **Top Attractions** — A bullet list of 5-8 must-visit attractions
   related to the traveler's interests, with a one-line description each.

4. **Food Recommendations** — A bullet list of must-try local dishes
   and 3-5 recommended restaurants/food spots, mentioning budget-friendly
   and mid-range options.

5. **Travel Tips** — Practical tips covering:
   - Best way to get around (local transport)
   - Money-saving tips to stay within budget
   - Safety or cultural etiquette tips
   - Best time of day/season notes if relevant

Keep the tone friendly and practical. Make sure the plan realistically
fits within the stated budget of {trip['budget']} for {trip['days']} days.
"""
    return prompt.strip()


## CELL 5: Call Gemini and get the itinerary ---------------------------
def generate_itinerary(client, trip):
    """
    Sends the prompt to Gemini using the new google-genai client and
    returns the generated itinerary text.
    """
    prompt = build_prompt(trip)
    print("\nGenerating your personalized travel plan... please wait...\n")

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
    )

    # response.text holds the generated Markdown content
    return response.text


## CELL 6: Main driver function -----------------------------------------
def main():
    client = configure_gemini()
    trip = get_user_inputs()
    itinerary = generate_itinerary(client, trip)

    print("\n" + "=" * 70)
    print(f" YOUR {trip['days']}-DAY TRAVEL PLAN FOR {trip['destination'].upper()}")
    print("=" * 70 + "\n")
    print(itinerary)

    # Optional: save the itinerary to a text file in Colab's file system
    save = input("\nSave this itinerary to a .txt file? (y/n): ").strip().lower()
    if save == "y":
        filename = f"{trip['destination'].replace(' ', '_')}_itinerary.txt"
        with open(filename, "w", encoding="utf-8") as f:
            f.write(itinerary)
        print(f"Saved as '{filename}'. Find it in the Colab file browser (left sidebar).")


## CELL 7: Run the planner -----------------------------------------------
if __name__ == "__main__":
    main()

Enter your Gemini API key: ··········

===== Smart Travel Planner: Tell us about your trip =====

Enter your destination (e.g., Goa, Paris, Tokyo): Tokyo
Enter your total budget (e.g., '₹30,000' or '$1500'): 30,000
Enter number of days for the trip (e.g., 5): 4
Enter your interests, comma-separated (e.g., history, food, nightlife, nature, shopping, adventure): nightlife

Generating your personalized travel plan... please wait...


 YOUR 4-DAY TRAVEL PLAN FOR TOKYO

Welcome to Tokyo! This exciting city offers a perfect blend of ultra-modern neon-lit streets, rich culture, and some of the best nightlife in the world. 

With a budget of **30,000 JPY (approx. $200 USD)** for your 4-day trip (allocating this for daily expenses: food, drinks, local transport, and entry fees, assuming flights and accommodation are pre-paid), you can experience an incredible, action-packed Tokyo adventure. This plan is specifically tailored to maximize your budget while diving headfirst into Tokyo's legendary 